# 🔧 Support Vector Machines
**ISLP Chapter 9 · Data Pattern Recognition for the Rest of Us**

> SVMs find the hyperplane that maximally separates classes. With kernels, they handle nonlinear boundaries. They work well in high dimensions and with imbalanced classes — making them a natural fit for fraud detection.

### Dataset
**Credit Card Fraud Detection**
Synthetic dataset modelled on the properties of the real European credit card fraud dataset (Kaggle). 284,807 transactions, 0.17% fraud rate — a classic imbalanced classification challenge in financial services.

### Time: ~60 minutes

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#f8f9fa',
    'axes.grid':True,'grid.alpha':0.4,'axes.spines.top':False,'axes.spines.right':False,'font.size':11})
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                              ConfusionMatrixDisplay, roc_auc_score, RocCurveDisplay)
from sklearn.datasets import make_classification
print("\u2713 Setup complete")

In [ ]:
# Credit Card Fraud — realistic synthetic dataset
# Properties match the real Kaggle dataset:
# - Highly imbalanced (0.17% fraud)
# - PCA-transformed features (V1-V8) + Amount + Time
np.random.seed(42)
n_legit, n_fraud = 10000, 17  # ~0.17% fraud rate

# Legitimate transactions
legit = np.random.multivariate_normal(
    mean=np.zeros(8),
    cov=np.eye(8) + 0.2*np.random.rand(8,8),
    size=n_legit)
legit_amount = np.random.lognormal(4.5, 1.2, n_legit)

# Fraudulent transactions — different distribution
fraud = np.random.multivariate_normal(
    mean=[2,-2,1.5,-1.5,0.8,-0.8,1,-0.5],
    cov=np.eye(8)*0.5,
    size=n_fraud)
fraud_amount = np.random.lognormal(3.8, 0.8, n_fraud)

X_raw = np.vstack([legit, fraud])
amounts = np.concatenate([legit_amount, fraud_amount])
y_fraud = np.array([0]*n_legit + [1]*n_fraud)

# Scale amount (like the real dataset)
scaler_amt = StandardScaler()
X_full = np.column_stack([X_raw, scaler_amt.fit_transform(amounts.reshape(-1,1))])

print(f"Credit card transactions: {len(y_fraud):,}")
print(f"Legitimate: {n_legit:,}  ({n_legit/len(y_fraud)*100:.1f}%)")
print(f"Fraudulent: {n_fraud:,}  ({n_fraud/len(y_fraud)*100:.2f}%)")
print(f"\nClass imbalance ratio: {n_legit/n_fraud:.0f}:1")
print("\nThis is why accuracy is a TERRIBLE metric here:")
print(f"  Always predicting 'legit' gives {n_legit/len(y_fraud)*100:.2f}% accuracy")
print("  but catches exactly 0% of fraud!")

## 🎯 Part 1 — The Maximal Margin Classifier

SVM finds the hyperplane that **maximizes the margin** — the distance to the nearest training points (support vectors) from each class.

```
Maximize M  subject to:  yᵢ(β₀ + β₁xᵢ₁ + ... + βₚxᵢₚ) ≥ M
```

The **soft margin** (C parameter) allows some misclassification:
- Large C → narrow margin, few violations, may overfit
- Small C → wide margin, tolerates more violations, more robust

For fraud detection: we want high **recall on fraud** (catch as many as possible) even at the cost of some false alarms.

In [ ]:
# Use 2 most discriminating features for visualization
X_tr, X_te, y_tr, y_te = train_test_split(X_full, y_fraud, test_size=0.25,
                                             random_state=42, stratify=y_fraud)
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s  = scaler.transform(X_te)

# Visualize decision boundary using top 2 features
X_2d = X_tr_s[:, :2]
y_2d = y_tr

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
C_vals   = [0.1, 1.0, 100.0]
titles   = ['Small C=0.1\n(wide margin, tolerant)',
            'C=1.0\n(balanced)',
            'Large C=100\n(tight margin, strict)']

for ax, C, title in zip(axes, C_vals, titles):
    svm = SVC(kernel='rbf', C=C, probability=True, class_weight='balanced')
    svm.fit(X_2d, y_2d)

    xx, yy = np.meshgrid(np.linspace(-4,4,200), np.linspace(-4,4,200))
    Z = svm.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.2, colors=['#1e5fa8','#e85d20'])
    ax.scatter(X_2d[y_2d==0,0], X_2d[y_2d==0,1], color='#1e5fa8',
               alpha=0.3, s=8, label='Legit')
    ax.scatter(X_2d[y_2d==1,0], X_2d[y_2d==1,1], color='#e85d20',
               s=80, marker='x', lw=2, label='Fraud')

    fraud_recall = svm.score(X_2d[y_2d==1], y_2d[y_2d==1])
    ax.set_title(f'{title}\nFraud recall={fraud_recall:.0%}', fontsize=10)
    ax.legend(fontsize=8)

plt.suptitle('SVM Decision Boundaries — Credit Card Fraud\n'
             'Orange X marks = actual fraud transactions',
             fontsize=12, y=1.02)
plt.tight_layout(); plt.show()

## 🔮 Part 2 — The Kernel Trick

For nonlinear fraud patterns, the **RBF (Radial Basis Function) kernel** maps data to a higher-dimensional space where a linear boundary separates classes.

```
K(x,z) = exp(-γ||x-z||²)
```
γ controls how far a single training example's influence reaches — higher γ = tighter fit.

In [ ]:
# Full model with all features — RBF kernel, class_weight to handle imbalance
svm_full = SVC(kernel='rbf', C=1.0, gamma='scale',
               class_weight='balanced', probability=True, random_state=42)
svm_full.fit(X_tr_s, y_tr)
y_pred = svm_full.predict(X_te_s)
y_prob = svm_full.predict_proba(X_te_s)[:,1]

print("=== SVM on Credit Card Fraud ===\n")
print(classification_report(y_te, y_pred, target_names=['Legit','Fraud'],
                            digits=4))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

cm = confusion_matrix(y_te, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['Legit','Fraud']).plot(
    ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix\n(we care most about fraud recall)')

RocCurveDisplay.from_estimator(svm_full, X_te_s, y_te, ax=axes[1])
axes[1].set_title(f'ROC Curve (AUC={roc_auc_score(y_te,y_prob):.4f})')
plt.tight_layout(); plt.show()

n_sv = svm_full.n_support_.sum()
print(f"\n\u2714 Number of support vectors: {n_sv}")
print(f"   These are the only transactions that define the boundary")
print(f"   All other transactions can be removed without changing the model")

In [ ]:
# Business framing: threshold tuning
# In fraud detection, missing fraud (false negative) is far more costly than
# a false alarm (false positive). Adjust threshold to maximize fraud recall.
from sklearn.metrics import precision_recall_curve

precision, recall, thresholds = precision_recall_curve(y_te, y_prob)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(thresholds, precision[:-1], color='#1e5fa8', lw=2, label='Precision')
ax.plot(thresholds, recall[:-1],    color='#e85d20', lw=2, label='Recall (fraud caught)')
ax.axvline(0.5, color='#888', lw=1.5, ls='--', label='Default threshold = 0.5')
ax.set_xlabel('Classification Threshold')
ax.set_ylabel('Score')
ax.set_title('Precision-Recall Tradeoff\n'
             'Lower threshold = catch more fraud but more false alarms')
ax.legend()
plt.tight_layout(); plt.show()

print("\n\u2714 Business decision: fraud cost >> false alarm cost")
print("   A threshold of 0.2-0.3 may be more appropriate than 0.5")
print("   This is a BUSINESS decision, not a modeling one")

In [ ]:
# @title 📝 Quiz — Support Vector Machines
# @markdown Answer each question in the box below, then run the AI grading cell.

# @markdown **Q1:** What are support vectors and why do only they define the boundary?
# @markdown **Q2:** What does the C parameter control and what happens at extremes?
# @markdown **Q3:** Why is accuracy a bad metric for fraud detection?
# @markdown **Q4:** What is the kernel trick and why does RBF work well for fraud?
# @markdown **Q5:** Should you lower the fraud threshold from 0.5? Justify your answer.

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

# Collect into answers dict for grading cell
answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
missing = [k for k,v in answers.items() if not str(v).strip()]
print(f"  {5-len(missing)}/5 answered — run the AI grading cell below for feedback")

### 📤 Submit for AI Grading

In [ ]:
_NB_NAME_ = "Support Vector Machines"
# @title 🤖 AI Feedback — enter your username and click ▶ Run
# @markdown **No API key needed.** AI grading runs free inside your Colab session.
# @markdown
# @markdown Enter your GitHub username, then click ▶ Run for question-by-question feedback.

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"e.g. jsmith42"}

# ── runs automatically — do not edit below ───────────────────
import json, textwrap, re as _re, time
_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

def _get_quiz_questions():
    """Pull question text from the quiz cell @markdown lines."""
    try:
        import ipynbname
        nb_path = ipynbname.path()
        with open(nb_path) as f:
            nb = json.load(f)
        for cell in nb["cells"]:
            src = "".join(cell.get("source", []))
            if "@title" in src and "Quiz" in src:
                qs = re.findall(r"@markdown \*\*Q\d+:\*\* (.+)", src)
                if qs: return qs
    except Exception:
        pass
    return []

def _call_gemini(prompt, max_retries=3):
    """Call Gemini with retry on 429 rate limit."""
    last_err = None
    for attempt in range(max_retries):
        try:
            import google.generativeai as genai
            import google.auth, google.auth.transport.requests
            creds, _ = google.auth.default()
            creds.refresh(google.auth.transport.requests.Request())
            genai.configure(credentials=creds)
            model  = genai.GenerativeModel("gemini-2.0-flash")
            result = model.generate_content(
                prompt,
                generation_config={"max_output_tokens": 1500, "temperature": 0.3}
            )
            return result.text, "Gemini via Colab"
        except Exception as e:
            last_err = str(e)
            if "429" in str(e) or "quota" in str(e).lower():
                wait = 2 ** attempt
                print(f"  Rate limit hit — waiting {wait}s before retry {attempt+1}/{max_retries}...")
                time.sleep(wait)
                continue
            break
    # Try stored key as fallback
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        if key and len(key) > 10:
            import google.generativeai as genai
            genai.configure(api_key=key)
            model  = genai.GenerativeModel("gemini-2.0-flash")
            result = model.generate_content(prompt)
            return result.text, "Gemini via key"
    except Exception:
        pass
    return None, last_err

def _build_prompt(answers_dict, nb_name, questions):
    answered  = {k: v for k, v in answers_dict.items() if str(v).strip()}
    n_total   = len(answers_dict)
    n_done    = len(answered)

    # Pair each question with the student answer
    qa_pairs = ""
    for i, (k, v) in enumerate(answers_dict.items(), 1):
        q_text = questions[i-1] if i-1 < len(questions) else f"Question {i}"
        a_text = str(v).strip() if str(v).strip() else "(no answer)"
        qa_pairs += f"Q{i}: {q_text}\nA{i}: {a_text}\n\n"

    return f"""You are a TA grading a student quiz for the "{nb_name}" notebook in a data science course called "Data Pattern Recognition for the Rest of Us" (based on ISLP and business analytics).

{qa_pairs.strip()}

For EACH question:
- Decide if the answer is CORRECT, PARTIALLY CORRECT, or INCORRECT
- A short paraphrase or reasonable approximation counts as CORRECT — do not require exact wording
- For INCORRECT or PARTIAL: name the specific concept they need to review (1 sentence)
- For CORRECT: brief confirmation of what they got right (1 sentence)

Then give an overall summary.

Respond ONLY in this exact JSON format (no markdown fences, no extra text):
{{
  "questions": [
    {{
      "q": 1,
      "status": "<CORRECT|PARTIAL|INCORRECT>",
      "comment": "<one specific sentence>"
    }}
  ],
  "quiz_score": <integer 0-{n_total}>,
  "grade": "<Excellent|Good|Needs Review|Incomplete>",
  "summary": "<2 sentences overall: strengths and what to revisit>",
  "review_tip": "<one specific concept, chapter, or notebook to review if they struggled — or 'Great work!' if excellent>"
}}

Scoring guide: CORRECT=1pt, PARTIAL=0.5pt (round to nearest int), INCORRECT=0pt."""

def _show(result, username, nb_name, source, questions):
    STATUS_ICON  = {"CORRECT": "\u2705", "PARTIAL": "\u26a0\ufe0f", "INCORRECT": "\u274c"}
    STATUS_COLOR = {"CORRECT": "\033[92m", "PARTIAL": "\033[93m", "INCORRECT": "\033[91m"}
    R = "\033[0m"
    grade = result.get("grade", "?")
    GRADE_COLOR = {"Excellent":"\033[92m","Good":"\033[94m",
                   "Needs Review":"\033[93m","Incomplete":"\033[91m"}
    GC = GRADE_COLOR.get(grade, "")
    n  = len(answers)
    qs = result.get("quiz_score", 0)
    bar = "\u2588"*qs + "\u2591"*(n-qs)

    print("\n" + "\u2550"*60)
    print(f"  \U0001f916 AI Feedback \u2014 {nb_name}")
    if source: print(f"  Powered by   {source}")
    print("\u2550"*60)
    print(f"  Student  : {'@'+username if username else '\u26a0 set GITHUB_USERNAME above'}")
    print(f"  Grade    : {GC}{grade}{R}")
    print(f"  Score    : {qs}/{n}  [{bar}]")
    print()

    # Question-by-question breakdown
    q_results = result.get("questions", [])
    if q_results:
        print("  \u2500"*29)
        for qr in q_results:
            idx    = qr.get("q", 0) - 1
            status = qr.get("status", "?")
            icon   = STATUS_ICON.get(status, "\u2753")
            color  = STATUS_COLOR.get(status, "")
            comment= qr.get("comment", "")
            q_text = questions[idx] if idx < len(questions) else f"Question {qr.get('q','?')}"
            # Truncate long question text for display
            q_short = q_text[:55] + "..." if len(q_text) > 55 else q_text
            print(f"  {icon} {color}Q{qr.get('q','?')} {status}{R}")
            print(f"     {q_short}")
            if comment:
                for line in textwrap.wrap(comment, 54):
                    print(f"     \u2192 {line}")
            print()

    # Summary
    summary = result.get("summary", "")
    if summary:
        print("  \u2500"*29)
        print("  Overall:")
        for line in textwrap.wrap(summary, 56):
            print(f"  {line}")

    # Review tip
    tip = result.get("review_tip", "")
    if tip and tip != "Great work!":
        print()
        for line in textwrap.wrap(f"\U0001f4d6 Review: {tip}", 56):
            print(f"  {line}")
    elif tip == "Great work!":
        print()
        print("  \U0001f4d6 Great work! Keep going.")

    print("\u2550"*60 + "\n")

def _fallback_grade(answers_dict):
    """Smart fallback — grade on completion only, no length penalty."""
    n  = len(answers_dict)
    nd = sum(1 for v in answers_dict.values() if str(v).strip())
    if nd == 0:
        return {"quiz_score":0,"grade":"Incomplete",
                "questions":[],
                "summary":"No answers provided — fill in the quiz above.",
                "review_tip":"Complete the quiz and re-run for AI feedback."}
    elif nd == n:
        return {"quiz_score":n,"grade":"Good",
                "questions":[],
                "summary":f"All {n} questions answered. AI grading was unavailable — re-run to get question-by-question feedback.",
                "review_tip":"Re-run this cell in a few minutes for detailed AI feedback."}
    else:
        return {"quiz_score":nd,"grade":"Needs Review",
                "questions":[],
                "summary":f"{nd}/{n} questions answered. Complete the remaining {n-nd} and re-run.",
                "review_tip":"Answer all questions for full feedback."}

# ── Main ──────────────────────────────────────────────────────
if "answers" not in globals():
    print("\u26a0  Run the quiz cell above first, then re-run this cell.")
else:
    nd       = sum(1 for v in answers.values() if str(v).strip())
    username = GITHUB_USERNAME.strip()
    questions = _get_quiz_questions()

    print(f"\n  Notebook : {_NB_TITLE}  \u2022  {nd}/{len(answers)} answered")
    if username: print(f"  Student  : @{username}")
    print(f"  Grading  : please wait...\n")

    prompt     = _build_prompt(answers, _NB_TITLE, questions)
    raw, src   = _call_gemini(prompt)

    if raw:
        try:
            clean  = _re.sub(r"```(?:json)?\s*|```","",raw).strip()
            result = json.loads(clean)
        except Exception:
            nd2    = sum(1 for v in answers.values() if str(v).strip())
            result = {"quiz_score":nd2,"grade":"Good" if nd2==len(answers) else "Needs Review",
                      "questions":[],"summary":raw[:400],"review_tip":""}
    else:
        if src: print(f"  \u26a0 Gemini unavailable ({src[:60]}) \u2014 showing completion feedback\n")
        result = _fallback_grade(answers)

    _show(result, username, _NB_TITLE, src if raw else None, questions)
    print("  \U0001f4be  File \u2192 Save a copy in GitHub \u2192 your fork\n")


---
*Data Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*